# Evaluación del escenario base

Consolida las métricas de los 4 modelos baseline en una sola tabla y calcula métricas ordinales homogéneas (`accuracy_adjacent`, `mae_ordinal`, `qwk`) para que la comparación con geo y extendido sea justa.

**Importante:** este notebook **re-entrena** los modelos baseline con los hiperparámetros ya seleccionados en sus notebooks originales. No modifica ni sobrescribe los artefactos baseline: guarda todo con el prefijo `base_eval_` y produce `reports/comparacion_base.csv`.

Si prefieres no re-entrenar, cambia `RECALCULAR_BASE = False`; en ese caso se leen los CSV/JSON existentes de `reports/` y las métricas ordinales quedan como `NaN`.

In [ ]:
import sys
sys.path.append('../..')

import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from src.model_features import evaluar_modelo_ordinal, metricas_a_fila
from src.model_lightgbm import construir_modelo_lightgbm
from src.model_ordinal import codificar_y, decodificar_y, entrenar, predecir
from src.model_random_forest import construir_modelo_random_forest
from src.model_xgboost import construir_modelo_xgboost
from src.preprocessing import descartar_columnas
from src.utils import (
    CLASES,
    REPORTS_DIR,
    SEED,
    cargar_dataset,
    fijar_semilla,
    split_estratificado,
)

sns.set_theme(style='whitegrid')
fijar_semilla(SEED)
print('Semilla:', SEED)

In [ ]:
RECALCULAR_BASE = True
COLUMNAS_METRICAS = [
    'accuracy',
    'balanced_accuracy',
    'f1_macro',
    'f1_weighted',
    'accuracy_adjacent',
    'mae_ordinal',
    'qwk',
]

In [ ]:
if RECALCULAR_BASE:
    df = descartar_columnas(cargar_dataset())
    print('Dataset base:', df.shape)

    X_train, X_val, X_test, y_train, y_val, y_test = split_estratificado(
        df,
        target='price_range',
        excluir=['price_clean', 'id'],
        seed=SEED,
    )
    print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

    modelos = {}

    modelos['random_forest'] = construir_modelo_random_forest(
        X_train,
        n_estimators=100,
        max_depth=None,
        max_features='sqrt',
    )
    modelos['random_forest'].fit(X_train, y_train)

    modelos['xgboost'] = construir_modelo_xgboost(X_train, learning_rate=0.1)
    modelos['xgboost'].fit(X_train, codificar_y(y_train))

    modelos['lightgbm'] = construir_modelo_lightgbm(X_train, num_leaves=63)
    modelos['lightgbm'].fit(X_train, y_train)

    modelos['ordinal_logistic'] = entrenar(X_train, y_train, alpha=0.1)

    filas = []
    for nombre, modelo in modelos.items():
        if nombre == 'ordinal_logistic':
            pred_val = predecir(modelo, X_val)
            pred_test = predecir(modelo, X_test)
        elif nombre == 'xgboost':
            pred_val = decodificar_y(modelo.predict(X_val))
            pred_test = decodificar_y(modelo.predict(X_test))
        else:
            pred_val = modelo.predict(X_val)
            pred_test = modelo.predict(X_test)

        met_val = evaluar_modelo_ordinal(y_val, pred_val)
        met_test = evaluar_modelo_ordinal(y_test, pred_test)
        print('=' * 70)
        print(nombre)
        print('Validación:', {k: round(v, 4) for k, v in met_val.items() if k != 'matriz_confusion'})
        print('Test:', {k: round(v, 4) for k, v in met_test.items() if k != 'matriz_confusion'})

        filas.append(metricas_a_fila(nombre, met_test, escenario='base'))

        ConfusionMatrixDisplay(
            met_test['matriz_confusion'], display_labels=CLASES
        ).plot(cmap='Blues', values_format='d')
        plt.title(f'{nombre} base - test')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(
            REPORTS_DIR / f'{nombre}_base_eval_matriz_confusion.png',
            dpi=150,
            bbox_inches='tight',
        )
        plt.close()

    base_df = pd.DataFrame(filas)
else:
    base_df = pd.DataFrame(
        [
            {
                'escenario': 'base',
                'modelo': 'random_forest',
                **pd.read_csv(REPORTS_DIR / 'random_forest_metricas.csv').iloc[0].to_dict(),
            },
            {
                'escenario': 'base',
                'modelo': 'xgboost',
                **pd.read_csv(REPORTS_DIR / 'xgboost_metricas_test.csv').iloc[0].to_dict(),
            },
            {
                'escenario': 'base',
                'modelo': 'lightgbm',
                **pd.read_csv(REPORTS_DIR / 'lightgbm_metricas_test.csv').iloc[0].to_dict(),
            },
            {
                'escenario': 'base',
                'modelo': 'ordinal_logistic',
                **json.load(open(REPORTS_DIR / 'ordinal_logistic_metricas.json', encoding='utf-8')),
            },
        ]
    )

base_df = base_df.sort_values('f1_macro', ascending=False).reset_index(drop=True)
for columna in COLUMNAS_METRICAS:
    if columna not in base_df.columns:
        base_df[columna] = float('nan')

display(base_df[['escenario', 'modelo'] + COLUMNAS_METRICAS])

In [ ]:
base_df.to_csv(REPORTS_DIR / 'comparacion_base.csv', index=False)
base_df.to_csv(REPORTS_DIR / 'base_metricas_completas.csv', index=False)

base_df.set_index('modelo')[['accuracy', 'f1_macro', 'accuracy_adjacent']].plot(
    kind='bar', figsize=(9, 5)
)
plt.title('Escenario base: comparación de modelos')
plt.ylabel('Métrica')
plt.ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'base_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

print('Guardado en:', REPORTS_DIR / 'comparacion_base.csv')

## Lectura de resultados

- Los valores deben coincidir (dentro del ruido de ±0.001) con los reportados en los notebooks baseline: RF ≈ 0.681, XGBoost ≈ 0.671–0.675, LightGBM ≈ 0.688, Ordinal ≈ 0.625 de accuracy.
- `accuracy_adjacent` suele ser mucho mayor que `accuracy` porque solo penaliza errores de dos o más rangos. Es la métrica adecuada para un problema ordinal.
- Esta tabla es la referencia contra la que se comparan geo y extendido.